# ALS Beamline Script Generation

This notebook demonstrates how to use the ALS beamline utilities to generate run files and macros for ALS beamline 11.0.1.2.

The utilities support:
- **PXR (Polarized X-ray Reflectivity)** scans
- **ESCAN (Energy scans)** for NEXAFS measurements
- **RSOXS (Resonant Soft X-ray Scattering)** scans
- **Macro generation** for motor movements, trajectories, and instrument setup

In [1]:
from pathlib import Path

from pyref.beamline import (
    add_comment,
    begin_macro,
    build_macro,
)
from pyref.beamline.pxr_widgets import PXR_ScriptGen
from pyref.beamline.rsoxs_widgets import RSOXS_ScriptGen

print("ALS beamline utilities imported successfully")

ModuleNotFoundError: No module named 'PyQt6'

## Polarized X-ray Reflectivity (PXR) Widget

The PXR widget creates scripts for polarized X-ray reflectivity measurements with configurable theta ranges, energies, and motor positions.

In [2]:
output_dir = Path(
    "/Volumes/DATA/Collins/2026Feb/.macro"
)

PXR_ScriptGen(
    path=str(output_dir)
)

### Using PXR Widget

1. **Configure experiment parameters**:
   - Set energy range (start, stop, delta)
   - Set theta end angle
   - Configure motor positions for each step (Sample Theta, Higher Order Suppressor, Exposure)

2. **Add scans**: Use the "Add Scan" button to create multiple reflectivity scans

3. **Configure each scan**:
   - Set theta values for each measurement point
   - Set Higher Order Suppressor positions
   - Set exposure times for each point

4. **Save scripts**: Use "Save CCD Scan File" to generate the run file

### Example Configuration

Default PXR parameters:
- Energy range: 250-280 eV
- Theta end: 60 degrees
- Sample Theta steps: [1, 5, 10, 15, 20, 30, 40] degrees
- Higher Order Suppressor: [10, 9, 8.5, 7.5, 7.5, 7.5, 7.5]
- Exposure times: [0.0015, 0.0015, 0.0015, 0.0015, 0.1, 0.5, 1] seconds

## Resonant Soft X-ray Scattering (RSOXS) Widget

The RSOXS widget creates scripts for resonant soft X-ray scattering measurements with configurable sample positions and energy scans.

In [ ]:
rsoxs_experiment = RSOXS_ScriptGen(
    exp_name="RSOXS Scattering",
    save_path=str(output_dir)
)

display(rsoxs_experiment.GUI)

### Using RSOXS Widget

1. **Set sample name**: Enter the name of your sample

2. **Configure scan details**:
   - Select energy scan type (CARBON_ESCAN, QUICK_ESCAN, or CUSTOM)
   - Set Sample X, Y, and Theta positions
   - Configure energy scan steps if using CUSTOM

3. **Add multiple scans**: Use the interface to add multiple measurement positions

4. **Save scripts**: Generate run files for the beamline

### Example Configuration

Default RSOXS parameters:
- Sample Theta: 90 degrees (typical for RSOXS)
- Energy scan: CARBON_ESCAN (270-370 eV with variable step sizes)
- Sample X, Y: 0 (adjust as needed for your sample)

## Advanced: Programmatic Macro Generation

You can also generate macros programmatically without using the widgets.

In [ ]:
from pyref.beamline.beamline_scan_macros import (
    XRR_lineup,
    analog_from_file,
    run_dict_trajectory,
    sample_photo,
)

macro = begin_macro("Full XRR Measurement")

macro += [add_comment("Initialize beamline")]
macro += run_dict_trajectory({
    "CCD Y": 100.0,
    "CCD Theta": 0.0,
    "CCD X": 100.5,
    "Beam Stop": 15.0
})

macro += [add_comment("Align sample")]
macro += XRR_lineup(repeat=3)

macro += [add_comment("Take sample photo")]
macro += sample_photo("sample_image")

macro += [add_comment("Run reflectivity scan from file")]
macro += [analog_from_file(
    name="xrr_scan",
    path_to_scan=str(output_dir / "scan_file.txt"),
    delay=0.1,
    count_time=0.5
)]

for step in macro:
    print(step)

In [ ]:
build_macro("full_xrr_measurement", str(output_dir), macro)
print(f"Macro saved to {output_dir / 'full_xrr_measurement.txt'}")

## Loading and Saving Configurations

All widgets support saving configurations to JSON and loading them later.

In [ ]:
import json

example_config = {
    "energy_start": 250.0,
    "energy_stop": 300.0,
    "theta_end": 60.0,
    "Sample Theta": [1, 5, 10, 15, 20, 30, 40],
    "Higher Order Suppressor": [10, 9, 8.5, 7.5, 7.5, 7.5, 7.5],
    "Exposure": [0.0015, 0.0015, 0.0015, 0.0015, 0.1, 0.5, 1]
}

config_file = output_dir / "pxr_config.json"
with open(config_file, "w") as f:
    json.dump(example_config, f, indent=2)

print(f"Example configuration saved to {config_file}")
print("\nConfiguration contents:")
print(json.dumps(example_config, indent=2))

## Trajectory Options

Predefined motor trajectories are available for common beamline configurations.

In [ ]:
from pyref.beamline.beamline_scan_macros import (
    CCDFar_LowEn,
    Photodiode_Far,
    XRR_init,
)

print("Available trajectory dictionaries:")
print("\n1. Photodiode_Far:")
print(json.dumps(Photodiode_Far, indent=2))

print("\n2. XRR_init:")
print(json.dumps(XRR_init, indent=2))

print("\n3. CCDFar_LowEn:")
print(json.dumps(CCDFar_LowEn, indent=2))

macro = begin_macro("Trajectory Example")
macro += run_dict_trajectory(CCDFar_LowEn)
macro += [add_comment("Now ready for low energy measurements")]

print("\n\nExample macro using trajectory:")
for step in macro:
    print(step)

## Summary

The ALS beamline utilities provide:

1. **Interactive widgets** for creating scan scripts:
   - ESCAN_ScriptGen for energy scans
   - PXR_Experiment for reflectivity measurements
   - RSOXS_Experiment for scattering measurements

2. **Macro generation functions** for:
   - Motor movements
   - Instrument setup
   - Trajectory execution
   - Sample alignment routines

3. **Configuration management**:
   - Save/load JSON configurations
   - Predefined trajectory options
   - Default scan parameters

4. **File output**:
   - Generate .txt run files for the beamline
   - Compatible with ALS beamline 11.0.1.2 control software

For more information, see the module documentation:
```python
from pyref.beamline import help
```